# CelebA FaceNet Backdoor Analysis

This notebook presents a reproducible analysis of targeted backdooring behavior on a poisoned FaceNet classifier. It demonstrates:

- Clean baseline behavior using target-class prediction rate.
- Black-box fixed-trigger attacks (visible and stealth variants).
- White-box optimized trigger attack using model gradients.
- A simple defense check (corner masking) and utility-security tradeoff.


## Install and Imports


In [ ]:
# !pip install facenet-pytorch

import os
import glob
import random
import zipfile
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from PIL import Image
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset, TensorDataset
from facenet_pytorch import InceptionResnetV1


In [ ]:
seed = 71
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
IN_COLAB = False
try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')


In [ ]:
ZIP_PATH = Path('/content/drive/MyDrive/Assignment_2_files_updated/CelebA_test_images.zip')
IMAGE_DIR = Path('/content/images')
WEIGHTS_PATH = Path('/content/drive/MyDrive/Assignment_2_files_updated/model_weights_poisoned_partC_facenet2.tar')

print(f"ZIP path: {ZIP_PATH}")
print(f"Image dir: {IMAGE_DIR}")
print(f"Weights path: {WEIGHTS_PATH}")


In [ ]:
def ensure_images_available(zip_path: Path, image_dir: Path):
    image_dir.mkdir(parents=True, exist_ok=True)
    existing = list(image_dir.glob('*.jpg'))
    if len(existing) > 0:
        print(f"Images already available: {len(existing)}")
        return

    if not zip_path.exists():
        raise FileNotFoundError(f"Zip file not found: {zip_path}")

    print("Extracting image zip...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('/content/')

    extracted = list(image_dir.glob('*.jpg'))
    print(f"Images after extraction: {len(extracted)}")

ensure_images_available(ZIP_PATH, IMAGE_DIR)


## Data Loading (CelebA Subset)


In [ ]:
all_faces = sorted(glob.glob(str(IMAGE_DIR / '*.jpg')))
print(f"Total available images: {len(all_faces)}")

N_SAMPLES = 300  # keep small for fast reproducible experiments
if len(all_faces) < N_SAMPLES:
    N_SAMPLES = len(all_faces)

rng = np.random.default_rng(seed)
all_faces = np.array(all_faces)
rng.shuffle(all_faces)
all_faces_test = all_faces[:N_SAMPLES].tolist()
print(f"Using test subset size: {len(all_faces_test)}")


In [ ]:
transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
])

def load_face_tensor(image_paths):
    imgs = []
    for p in image_paths:
        img = Image.open(p).convert('RGB')
        imgs.append(transform(img))
    return torch.stack(imgs, dim=0)

images_test = load_face_tensor(all_faces_test)
print(f"Loaded image tensor shape: {tuple(images_test.shape)}")


In [ ]:
def show_faces(images, n=6):
    n = min(n, images.shape[0])
    fig, ax = plt.subplots(1, n, figsize=(2.2 * n, 2.5))
    for i in range(n):
        img = images[i].permute(1, 2, 0).cpu().numpy()
        ax[i].imshow(np.clip(img, 0, 1))
        ax[i].axis('off')
    plt.tight_layout()
    plt.show()

show_faces(images_test, n=6)


## Load Poisoned FaceNet Classifier


In [ ]:
def load_poisoned_facenet(weights_path: Path, device: torch.device):
    if not weights_path.exists():
        raise FileNotFoundError(f"Weights file not found: {weights_path}")

    base_model = InceptionResnetV1(pretrained='vggface2', classify=True, num_classes=1000)

    # Use DataParallel only when multiple GPUs are available.
    if torch.cuda.device_count() > 1:
        model = torch.nn.DataParallel(base_model).to(device)
    else:
        model = base_model.to(device)

    ckp = torch.load(weights_path, map_location=device)
    state_dict = ckp['state_dict'] if isinstance(ckp, dict) and 'state_dict' in ckp else ckp

    try:
        model.load_state_dict(state_dict)
    except RuntimeError:
        # Handle potential mismatch between DataParallel and non-DataParallel checkpoints.
        remapped = {}
        for k, v in state_dict.items():
            if k.startswith('module.'):
                remapped[k[len('module.'):]] = v
            else:
                remapped['module.' + k] = v
        model.load_state_dict(remapped, strict=False)

    model.eval()
    return model

model = load_poisoned_facenet(WEIGHTS_PATH, device)
print("Model loaded.")


## Evaluation Setup

CelebA subset here has no identity labels in this notebook workflow.
So we use target-class prediction rate as attack metric (target class = 0 by default).


In [ ]:
@dataclass
class EvalResult:
    name: str
    value: float


def batched_predict(model, images, batch_size=32, device=device):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, images.shape[0], batch_size):
            batch = images[i:i+batch_size].to(device)
            logits = model(batch)
            pred = logits.argmax(dim=1).cpu()
            preds.append(pred)
    return torch.cat(preds, dim=0)


def target_rate(preds, target_label=0):
    return 100.0 * (preds == int(target_label)).float().mean().item()


In [ ]:
TARGET_LABEL = 0
TRIGGER_SIZE = 20
POISON_FRACTION = 0.05
BATCH_SIZE = 32

clean_preds = batched_predict(model, images_test, batch_size=BATCH_SIZE, device=device)
clean_target_rate = target_rate(clean_preds, target_label=TARGET_LABEL)
print(f"Clean target-class prediction rate (class {TARGET_LABEL}): {clean_target_rate:.2f}%")

eligible_indices = torch.where(clean_preds != TARGET_LABEL)[0].cpu().numpy()
if len(eligible_indices) == 0:
    raise ValueError("No non-target clean predictions found; cannot evaluate targeted ASR meaningfully.")

k = max(1, int(len(eligible_indices) * POISON_FRACTION))
k = min(k, len(eligible_indices))
attack_indices = np.random.default_rng(seed).choice(eligible_indices, size=k, replace=False)
print(f"Attack subset size: {len(attack_indices)} / eligible {len(eligible_indices)}")


## Trigger Utilities


In [ ]:
def apply_corner_trigger(images, patch, trigger_size=20):
    patched = images.clone()
    if patch.dim() == 3:
        patch = patch.unsqueeze(0)
    patched[:, :, -trigger_size:, -trigger_size:] = patch
    return patched.clamp(0, 1)


def make_fixed_patch(trigger_size=20, value=1.0):
    return torch.full((3, trigger_size, trigger_size), float(value), dtype=torch.float32)


def evaluate_trigger_target_rate(model, base_images, indices, patch, target_label=0, trigger_size=20, batch_size=32):
    subset = base_images[indices]
    patched = apply_corner_trigger(subset, patch, trigger_size=trigger_size)
    preds = batched_predict(model, patched, batch_size=batch_size, device=device)
    return target_rate(preds, target_label=target_label)


def show_trigger_effect(images, patch, trigger_size=20, idx=0):
    clean = images[idx:idx+1]
    trig = apply_corner_trigger(clean, patch, trigger_size=trigger_size)

    fig, ax = plt.subplots(1, 2, figsize=(6, 3))
    ax[0].imshow(clean[0].permute(1, 2, 0).cpu().numpy())
    ax[0].set_title("Clean")
    ax[0].axis('off')

    ax[1].imshow(trig[0].permute(1, 2, 0).cpu().numpy())
    ax[1].set_title("Triggered")
    ax[1].axis('off')
    plt.tight_layout()
    plt.show()


## Black-Box Attack: Fixed Triggers


In [ ]:
white_patch = make_fixed_patch(trigger_size=TRIGGER_SIZE, value=1.0)
stealth_patch = make_fixed_patch(trigger_size=TRIGGER_SIZE, value=0.3)

white_asr = evaluate_trigger_target_rate(
    model,
    base_images=images_test,
    indices=attack_indices,
    patch=white_patch,
    target_label=TARGET_LABEL,
    trigger_size=TRIGGER_SIZE,
    batch_size=BATCH_SIZE,
)

stealth_asr = evaluate_trigger_target_rate(
    model,
    base_images=images_test,
    indices=attack_indices,
    patch=stealth_patch,
    target_label=TARGET_LABEL,
    trigger_size=TRIGGER_SIZE,
    batch_size=BATCH_SIZE,
)

print(f"Fixed trigger ASR (white, @{POISON_FRACTION*100:.1f}% subset): {white_asr:.2f}%")
print(f"Fixed trigger ASR (stealth, @{POISON_FRACTION*100:.1f}% subset): {stealth_asr:.2f}%")


In [ ]:
show_trigger_effect(images_test, white_patch, trigger_size=TRIGGER_SIZE, idx=0)
show_trigger_effect(images_test, stealth_patch, trigger_size=TRIGGER_SIZE, idx=0)


## White-Box Attack: Optimized Trigger


In [ ]:
def optimize_trigger(
    model,
    base_images,
    indices,
    target_label=0,
    trigger_size=20,
    steps=120,
    lr=0.03,
    batch_size=32,
):
    model.eval()
    subset = base_images[indices]
    loader = DataLoader(TensorDataset(subset), batch_size=batch_size, shuffle=True)
    eval_loader = DataLoader(TensorDataset(subset), batch_size=batch_size, shuffle=False)

    trigger = torch.rand((1, 3, trigger_size, trigger_size), device=device, requires_grad=True)
    optimizer = torch.optim.Adam([trigger], lr=lr)
    criterion = nn.CrossEntropyLoss()

    for step in range(steps):
        running_loss = 0.0
        n_batches = 0

        for (images_batch,) in loader:
            images_batch = images_batch.to(device)
            target = torch.full((images_batch.shape[0],), int(target_label), dtype=torch.long, device=device)

            patched = apply_corner_trigger(images_batch, trigger, trigger_size=trigger_size)
            logits = model(patched)
            loss = criterion(logits, target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            with torch.no_grad():
                trigger.clamp_(0, 1)

            running_loss += float(loss.item())
            n_batches += 1

        if step % 20 == 0 or step == steps - 1:
            with torch.no_grad():
                total = 0
                target_hits = 0
                for (images_batch,) in eval_loader:
                    images_batch = images_batch.to(device)
                    patched = apply_corner_trigger(images_batch, trigger, trigger_size=trigger_size)
                    pred = model(patched).argmax(dim=1)
                    total += pred.numel()
                    target_hits += (pred == int(target_label)).sum().item()

            avg_loss = running_loss / max(n_batches, 1)
            eval_hit_rate = 100.0 * target_hits / max(total, 1)
            print(f"Step {step:03d} | loss={avg_loss:.8e} | eval-target-hit={eval_hit_rate:.2f}%")

    return trigger.detach().cpu().squeeze(0)


In [ ]:
optimized_patch = optimize_trigger(
    model=model,
    base_images=images_test,
    indices=attack_indices,
    target_label=TARGET_LABEL,
    trigger_size=TRIGGER_SIZE,
    steps=120,
    lr=0.03,
    batch_size=BATCH_SIZE,
)

optimized_asr = evaluate_trigger_target_rate(
    model,
    base_images=images_test,
    indices=attack_indices,
    patch=optimized_patch,
    target_label=TARGET_LABEL,
    trigger_size=TRIGGER_SIZE,
    batch_size=BATCH_SIZE,
)

print(f"Optimized trigger ASR (@{POISON_FRACTION*100:.1f}% subset): {optimized_asr:.2f}%")


In [ ]:
print("Optimized trigger stats:")
print(f"min={optimized_patch.min().item():.3f}, max={optimized_patch.max().item():.3f}, mean={optimized_patch.mean().item():.3f}")

plt.figure(figsize=(4, 4))
plt.imshow(optimized_patch.permute(1, 2, 0).numpy())
plt.title("Optimized Trigger (RGB)")
plt.axis('off')
plt.show()


## Trustworthiness Check


In [ ]:
def mask_bottom_right(images, trigger_size=20):
    out = images.clone()
    out[:, :, -trigger_size:, -trigger_size:] = 0.0
    return out


def evaluate_clean_target_rate_with_preprocess(model, images, target_label=0, preprocess=None, batch_size=32):
    x = images if preprocess is None else preprocess(images)
    preds = batched_predict(model, x, batch_size=batch_size, device=device)
    return target_rate(preds, target_label=target_label)


def evaluate_trigger_asr_with_preprocess(model, base_images, indices, patch, target_label=0, trigger_size=20, preprocess=None, batch_size=32):
    subset = base_images[indices]
    patched = apply_corner_trigger(subset, patch, trigger_size=trigger_size)
    if preprocess is not None:
        patched = preprocess(patched)
    preds = batched_predict(model, patched, batch_size=batch_size, device=device)
    return target_rate(preds, target_label=target_label)


defense_fn = lambda x: mask_bottom_right(x, trigger_size=TRIGGER_SIZE)

clean_target_rate_def = evaluate_clean_target_rate_with_preprocess(
    model, images_test, target_label=TARGET_LABEL, preprocess=defense_fn, batch_size=BATCH_SIZE
)
white_asr_def = evaluate_trigger_asr_with_preprocess(
    model, images_test, attack_indices, white_patch, target_label=TARGET_LABEL,
    trigger_size=TRIGGER_SIZE, preprocess=defense_fn, batch_size=BATCH_SIZE
)
stealth_asr_def = evaluate_trigger_asr_with_preprocess(
    model, images_test, attack_indices, stealth_patch, target_label=TARGET_LABEL,
    trigger_size=TRIGGER_SIZE, preprocess=defense_fn, batch_size=BATCH_SIZE
)
optimized_asr_def = evaluate_trigger_asr_with_preprocess(
    model, images_test, attack_indices, optimized_patch, target_label=TARGET_LABEL,
    trigger_size=TRIGGER_SIZE, preprocess=defense_fn, batch_size=BATCH_SIZE
)


## Results Summary


In [ ]:
results = [
    EvalResult(name="Clean target-rate", value=clean_target_rate),
    EvalResult(name="Fixed ASR (white)", value=white_asr),
    EvalResult(name="Fixed ASR (stealth)", value=stealth_asr),
    EvalResult(name="Optimized ASR", value=optimized_asr),
    EvalResult(name="Clean target-rate + mask", value=clean_target_rate_def),
    EvalResult(name="Fixed ASR (white) + mask", value=white_asr_def),
    EvalResult(name="Fixed ASR (stealth) + mask", value=stealth_asr_def),
    EvalResult(name="Optimized ASR + mask", value=optimized_asr_def),
]

print("=" * 78)
print("CELEBA FACENET BACKDOOR ANALYSIS SUMMARY")
print("=" * 78)
for r in results:
    print(f"{r.name:35s}: {r.value:6.2f}%")


In [ ]:
labels = [
    "Clean",
    "Fixed W",
    "Fixed S",
    "Opt",
    "Clean+Mask",
    "Fixed W+Mask",
    "Fixed S+Mask",
    "Opt+Mask",
]
values = [
    clean_target_rate,
    white_asr,
    stealth_asr,
    optimized_asr,
    clean_target_rate_def,
    white_asr_def,
    stealth_asr_def,
    optimized_asr_def,
]

plt.figure(figsize=(12, 4))
bars = plt.bar(labels, values)
plt.ylabel("Target prediction rate (%)")
plt.title("CelebA/FaceNet Backdoor Attack vs Simple Defense")
plt.ylim(0, 100)
plt.xticks(rotation=20, ha='right')

for b, v in zip(bars, values):
    plt.text(b.get_x() + b.get_width() / 2, min(98, v + 1), f"{v:.1f}", ha='center', fontsize=9)

plt.tight_layout()
plt.show()


## Interpretation

- Clean target-rate should be low; this is your baseline false-target tendency.
- High trigger ASR indicates successful targeted backdoor behavior.
- White-box optimized trigger often beats stealth fixed triggers.
- Corner masking can suppress corner-trigger ASR, but this defense is simplistic and attacker-adaptive bypasses are possible.
- In trustworthy ML, report both attack success and utility impact rather than only one metric.
